# Data Pipeline: Validation and Derived-Data Generation

This notebook sits **upstream** of `inference_frontier_analysis.ipynb`.
Run it first. It does not change curated input files.

## Data layers

| Layer | Meaning | Who writes it |
|---|---|---|
| **L0 curated** | Facts entered by hand from a cited source (prices, specs, scores). | `scripts/generate_data.py`, manual edits |
| **L1 validated** | L0 after the checks in this notebook pass. | This notebook (read-only checks) |
| **L2 derived** | Values computed from L0 with a formula (memory footprint, fit, TCO, task cost). | This notebook → `data/generated/` |

**Rule:** a derived number must never be typed by hand. If a value can be computed, compute it here.

## Severity levels

* `ERROR` – the value is wrong or the data is broken (bad key, impossible number, formula identity fails).
* `WARN` – the value is suspect or two datasets disagree.
* `INFO` – context for the reader. No action is required.

## Checks

1. Schema and value ranges
2. Primary-key uniqueness
3. Referential integrity (model, hardware, source keys)
4. Provenance audit
5. Physics and formula re-derivation
6. Cross-dataset consistency
7. Generate L2 datasets
8. Summary and gate

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

try:
    display  # noqa: F821  (defined inside Jupyter)
except NameError:  # plain-python execution
    display = print

_cwd = Path.cwd()
RESEARCH_DIR = _cwd if (_cwd / "data").is_dir() else _cwd / "research"
DATA_DIR = RESEARCH_DIR / "data"
GEN_DIR = DATA_DIR / "generated"
assert DATA_DIR.is_dir(), f"Data directory not found: {DATA_DIR}"

SNAPSHOT_DATE = pd.Timestamp("2026-10-11")
REL_TOL = 0.05          # 5% relative tolerance for re-derived values
WRITE_OUTPUTS = True    # write L2 files to data/generated/ (never overwrites L0)
STRICT = True           # True: raise if any ERROR finding exists (Strict gating enabled)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

FINDINGS: list[dict] = []


def record(check, dataset, severity, message, keys=None):
    # Store one finding. `keys` lists the affected row identifiers.
    keys_txt = "" if keys is None else ", ".join(map(str, keys))
    FINDINGS.append({
        "check": check,
        "dataset": dataset,
        "severity": severity,
        "message": message,
        "keys": keys_txt[:400],
    })


def show_findings(check):
    df = pd.DataFrame([f for f in FINDINGS if f["check"] == check])
    if df.empty:
        print(f"[{check}] no findings")
    else:
        order = {"ERROR": 0, "WARN": 1, "INFO": 2}
        df = df.sort_values("severity", key=lambda s: s.map(order))
        print(f"[{check}] {len(df)} findings: {df.severity.value_counts().to_dict()}")
        display(df.reset_index(drop=True))


def norm_key(value):
    # Normalise display names and keys so that 'Qwen 3.8-27B' == 'qwen3.8-27b'.
    return re.sub(r"[^a-z0-9]", "", str(value).lower())


def row_keys(df, cols):
    # Join key columns into 'a/b/c' strings. Safe on empty frames.
    if df.empty:
        return []
    return ["/".join(map(str, t)) for t in df[cols].itertuples(index=False, name=None)]


def split_sources(value):
    if pd.isna(value):
        return []
    return [s.strip() for s in str(value).split(";") if s.strip()]


print("DATA_DIR =", DATA_DIR)

## 0. Dataset registry

Each dataset has a layer, a primary key (`pk`), and the script that writes it.
A CSV in `data/` that is not in the registry is reported, so no file escapes validation.

In [ ]:
REGISTRY = {
    # ---- L0 curated --------------------------------------------------------
    "sources":                 dict(layer="L0", pk=["source_id"],                         writer="scripts/generate_data.py"),
    "api_prices":              dict(layer="L0", pk=["model_key", "venue"],                writer="scripts/generate_data.py + manual"),
    "api_prices_hosted":       dict(layer="L0", pk=["model_key", "provider"],             writer="scripts/generate_data.py"),
    "assumptions":             dict(layer="L0", pk=["param_id"],                          writer="scripts/generate_data.py"),
    "fact_checks":             dict(layer="L0", pk=["fact_id"],                           writer="manual + HF model card verification"),
    "benchmarks":              dict(layer="L0", pk=["model_key", "benchmark", "benchmark_version", "reasoning_setting"], writer="scripts/generate_data.py + manual"),
    "electricity":             dict(layer="L0", pk=["region", "sector", "period"],        writer="scripts/generate_data.py"),
    "gpu_rental":              dict(layer="L0", pk=["provider", "gpu", "pricing_model", "region"], writer="scripts/generate_data.py"),
    "hardware":                dict(layer="L0", pk=["hw_key"],                            writer="scripts/generate_data.py"),
    "open_weight_models":      dict(layer="L0", pk=["model_key"],                         writer="scripts/generate_data.py"),
    "quant_artifacts":         dict(layer="L0", pk=["model_key", "quant_name", "publisher"], writer="scripts/generate_data.py"),
    "serving_perf":            dict(layer="L0", pk=["model_key", "quant", "hw_key", "engine", "context_tokens", "concurrency"], writer="scripts/generate_data.py"),
    # ---- Community & Empirical Benchmarks ----------------------------------
    "recipes":                 dict(layer="L0", pk=["recipe_id"],                         writer="research/data/recipes.csv"),
    "experiments":             dict(layer="L0", pk=["experiment_id"],                     writer="research/data/experiments.csv"),
    # ---- L2 derived by script ----------------------------------------------
    "model_fit_matrix":        dict(layer="L2", pk=["model_key", "hardware_key", "operating_mode"], writer="scripts/build_model.py"),
    "task_costs_modeled":      dict(layer="L2", pk=["workload_id", "model_key", "venue"], writer="scripts/build_model.py"),
    "hardware_tco_modeled":    dict(layer="L2", pk=["hw_key", "utilization_pct"],         writer="scripts/build_model.py"),
    "provider_unit_economics": dict(layer="L2", pk=["provider_type", "model_served"],     writer="scripts/build_model.py"),
    # ---- L2 values typed by hand (should be generated) ---------------------
    "open_weights_ram_residency": dict(layer="L2-hand", pk=["model_key"],                  writer="manual (notebook build scripts)"),
    "chinese_models_hardware_performance_matrix": dict(layer="L2-hand", pk=["model", "hardware_config", "quantization"], writer="scripts/generate_chinese_models_analysis.py"),
}

DFS = {name: pd.read_csv(DATA_DIR / f"{name}.csv") for name in REGISTRY}

unregistered = sorted(p.stem for p in DATA_DIR.glob("*.csv") if p.stem not in REGISTRY)
for name in unregistered:
    record("registry", name, "WARN", "CSV exists in data/ but is not in REGISTRY; it is not validated.")
for name, meta in REGISTRY.items():
    if meta["layer"] == "L2-hand":
        record("registry", name, "WARN",
               "Derived values are typed by hand. Regenerate them from L0 inputs (see Section 7).")

registry_df = pd.DataFrame([
    {"dataset": n, "layer": m["layer"], "rows": len(DFS[n]), "cols": DFS[n].shape[1],
     "pk": " + ".join(m["pk"]), "writer": m["writer"],
     "has_source_id": "source_id" in DFS[n].columns}
    for n, m in REGISTRY.items()
])
display(registry_df)
show_findings("registry")

## 1. Schema and value ranges

* Primary-key columns and `source_id` must not be empty.
* `confidence` and `evidence_type` must use the allowed vocabulary.
* Prices, sizes, speeds, and power must not be negative.
* Percent scores must be in $[0, 100]$; probabilities in $[0, 1]$.
* Dates must parse and must not be after the snapshot date.

In [ ]:
ENUMS = {
    "confidence": {"high", "medium", "low"},
    "evidence_type": {"official", "independent", "community", "derived"},
}
NONNEG_PATTERNS = ("_usd", "usd_", "_gb", "_tps", "_w", "_params_b", "_tflops", "context", "tokens")
DATE_COLS = ("release_date", "retrieved_utc", "price_date")

for name, df in DFS.items():
    meta = REGISTRY[name]

    missing_pk = [c for c in meta["pk"] if c not in df.columns]
    if missing_pk:
        record("schema", name, "ERROR", f"Primary-key columns missing: {missing_pk}")
        continue

    required = meta["pk"] + (["source_id"] if "source_id" in df.columns else [])
    for col in required:
        n_null = int(df[col].isna().sum())
        if n_null:
            record("schema", name, "ERROR", f"{n_null} empty values in required column '{col}'")

    if meta["layer"] == "L0" and "source_id" not in df.columns and name != "assumptions":
        record("schema", name, "WARN", "Curated dataset has no source_id column.")

    for col, allowed in ENUMS.items():
        if col in df.columns:
            bad = sorted(set(df[col].dropna()) - allowed)
            if bad:
                record("schema", name, "ERROR", f"'{col}' has values outside {sorted(allowed)}: {bad}")

    for col in df.select_dtypes(include="number").columns:
        if any(p in col for p in NONNEG_PATTERNS) and "headroom" not in col:  # headroom < 0 means a deficit
            neg = row_keys(df[df[col] < 0], meta["pk"])
            if neg:
                record("schema", name, "ERROR", f"Negative values in '{col}'", neg)

    if {"score", "unit"} <= set(df.columns):
        pct = df["unit"].astype(str).str.contains("%")
        out = df[pct & ~df["score"].between(0, 100)]
        if len(out):
            record("schema", name, "ERROR", "Percent score outside [0, 100]", out["model_key"].tolist())
    for col in [c for c in df.columns if c.endswith("_pct") and "util" not in c and "margin" not in c]:
        if pd.api.types.is_numeric_dtype(df[col]):
            out = df[~df[col].between(0, 100) & df[col].notna()]
            if len(out):
                record("schema", name, "ERROR", f"'{col}' outside [0, 100]", out.iloc[:, 0].tolist())
    if "task_success_rate" in df.columns:
        out = df[~df["task_success_rate"].between(0, 1)]
        if len(out):
            record("schema", name, "ERROR", "task_success_rate outside [0, 1]", out.iloc[:, 0].tolist())

    for col in DATE_COLS:
        if col in df.columns:
            parsed = pd.to_datetime(df[col], errors="coerce", utc=True).dt.tz_localize(None)
            bad = df.loc[parsed.isna() & df[col].notna(), meta["pk"][0]].tolist()
            if bad:
                record("schema", name, "ERROR", f"Unparseable dates in '{col}'", bad)
            future = df.loc[parsed > SNAPSHOT_DATE + pd.Timedelta(days=1), meta["pk"][0]].tolist()
            if future:
                record("schema", name, "ERROR", f"'{col}' is after the snapshot date {SNAPSHOT_DATE.date()}", future)

show_findings("schema")

## 2. Primary-key uniqueness

Two rows with the same primary key mean one fact is stored twice, possibly with different values.

In [ ]:
for name, df in DFS.items():
    pk = REGISTRY[name]["pk"]
    if not set(pk) <= set(df.columns):
        continue
    dup = df[df.duplicated(pk, keep=False)]
    if len(dup):
        keys = list(dict.fromkeys(row_keys(dup, pk)))
        record("pk", name, "ERROR", f"{len(dup)} rows share {len(keys)} primary keys {pk}", keys)

show_findings("pk")

## 3. Referential integrity

* Every `model_key` must exist in the model universe (`api_prices` ∪ `api_prices_hosted` ∪ `open_weight_models`).
* Every open-weight model in the residency table should have an architecture row in `open_weight_models`. Without it, the KV-cache size cannot be computed.
* Every hardware key must exist in `hardware.csv`.
* Every cited `source_id` must exist in `sources.csv`.

In [ ]:
MODEL_UNIVERSE = (set(DFS["api_prices"].model_key)
                  | set(DFS["api_prices_hosted"].model_key)
                  | set(DFS["open_weight_models"].model_key))
HW_KEYS = set(DFS["hardware"].hw_key)
SOURCE_IDS = set(DFS["sources"].source_id)

MODEL_REFS = {
    "benchmarks": "model_key",
    "open_weights_ram_residency": "model_key",
    "quant_artifacts": "model_key",
    "serving_perf": "model_key",
    "task_costs_modeled": "model_key",
    "model_fit_matrix": "model_key",
    "provider_unit_economics": "model_served",
}
for name, col in MODEL_REFS.items():
    orphans = sorted(set(DFS[name][col]) - MODEL_UNIVERSE)
    if orphans:
        canon = {norm_key(k): k for k in MODEL_UNIVERSE}
        hints = [f"{o} (did you mean {canon[norm_key(o)]}?)" if norm_key(o) in canon else o for o in orphans]
        record("refs", name, "ERROR", f"{len(orphans)} {col} values not in the model universe", hints)

no_arch = sorted(set(DFS["open_weights_ram_residency"].model_key) - set(DFS["open_weight_models"].model_key))
if no_arch:
    record("refs", "open_weights_ram_residency", "WARN",
           f"{len(no_arch)} of {len(DFS['open_weights_ram_residency'])} models have no architecture row "
           "in open_weight_models (KV cache cannot be derived)", no_arch)

# Display-name tables: map names to keys.
NAME_TO_KEY = {norm_key(k): k for k in MODEL_UNIVERSE | set(DFS["open_weights_ram_residency"].model_key)}
NAME_TO_KEY.update({"glm53flash": "glm-5.3", "yi1534b": "yi-1.5-34b"})
cm = DFS["chinese_models_hardware_performance_matrix"]
cm_unmapped = sorted({m for m in cm.model if norm_key(m) not in NAME_TO_KEY})
if cm_unmapped:
    record("refs", "chinese_models_hardware_performance_matrix", "WARN",
           "Model display names that do not map to a model_key", cm_unmapped)

HW_REFS = {"serving_perf": "hw_key", "hardware_tco_modeled": "hw_key", "model_fit_matrix": "hardware_key"}
for name, col in HW_REFS.items():
    orphans = sorted(set(DFS[name][col]) - HW_KEYS)
    if orphans:
        record("refs", name, "ERROR", f"{col} values not in hardware.csv", orphans)

for name, df in DFS.items():
    if "source_id" not in df.columns:
        continue
    cited = {s for v in df.source_id for s in split_sources(v)}
    unknown = sorted(cited - SOURCE_IDS)
    if unknown:
        record("refs", name, "ERROR", "Cited source_id not in sources.csv", unknown)

show_findings("refs")

## 4. Provenance audit

A number is only as good as its source. This section checks whether the provenance fields carry information.

* **Uniform confidence:** if every row is `high`, the column does not discriminate.
* **Self-citation:** `SRC-015` is this project's own ticker tape. It is a valid source for market prices and latency. It is **not** a valid source for model architecture, parameter counts, or benchmark scores.
* **Evidence mismatch:** a row marked `official` must cite at least one `official` source.
* **Batch timestamps:** many rows with one identical `retrieved_utc` suggest bulk entry, not per-fact retrieval.

In [ ]:
SELF_SOURCE = "SRC-015"
NON_MARKET_DATASETS = {"open_weight_models", "benchmarks", "quant_artifacts", "hardware"}
src_evidence = DFS["sources"].set_index("source_id")["evidence_type"].to_dict()

prov_rows = []
for name, df in DFS.items():
    if "source_id" not in df.columns:
        prov_rows.append({"dataset": name, "rows": len(df), "has_source_id": False})
        continue
    srcs = df.source_id.map(split_sources)
    only_self = srcs.map(lambda s: s == [SELF_SOURCE])
    any_self = srcs.map(lambda s: SELF_SOURCE in s)
    conf = df["confidence"].value_counts(normalize=True).to_dict() if "confidence" in df else {}
    prov_rows.append({
        "dataset": name, "rows": len(df), "has_source_id": True,
        "pct_high_conf": round(100 * conf.get("high", 0), 1),
        "pct_cites_self": round(100 * any_self.mean(), 1),
        "pct_only_self": round(100 * only_self.mean(), 1),
        "distinct_retrieved_utc": df["retrieved_utc"].nunique() if "retrieved_utc" in df else None,
    })

    if "confidence" in df and df["confidence"].nunique() == 1 and len(df) > 3:
        record("provenance", name, "WARN",
               f"All {len(df)} rows have confidence='{df['confidence'].iloc[0]}'; the field does not discriminate.")

    if name in NON_MARKET_DATASETS and only_self.any():
        key = REGISTRY[name]["pk"][0]
        record("provenance", name, "WARN",
               f"{int(only_self.sum())} rows cite only the project's own ticker tape ({SELF_SOURCE}) "
               "for non-market facts", df.loc[only_self, key].tolist())

    if "evidence_type" in df:
        official_rows = df["evidence_type"] == "official"
        has_official_src = srcs.map(lambda s: any(src_evidence.get(x) == "official" for x in s))
        mismatch = df[official_rows & ~has_official_src]
        if len(mismatch):
            key = REGISTRY[name]["pk"][0]
            record("provenance", name, "WARN",
                   "Rows marked evidence_type='official' cite no official source", mismatch[key].tolist())

    if "retrieved_utc" in df and len(df) >= 10 and df["retrieved_utc"].nunique() <= 2:
        record("provenance", name, "INFO",
               f"{len(df)} rows share {df['retrieved_utc'].nunique()} retrieval timestamp(s): bulk entry.")

rr = DFS["open_weights_ram_residency"]
record("provenance", "open_weights_ram_residency", "WARN",
       "No source_id column. Parameter counts in this table cannot be traced to a source.",
       rr.model_key.tolist())

display(pd.DataFrame(prov_rows))
show_findings("provenance")

## 5. Physics and formula re-derivation

Every derived value is recomputed from its inputs and compared with the stored value.

### 5a. Units

Model sizes use **decimal GB** ($10^9$ bytes), as model cards do.
Hardware memory is sold in **binary GiB** ($2^{30}$ bytes): a "512 GB" machine holds $512 \times 1.0737 = 549.8$ decimal GB.
Mixing the two gives a 7% error at every tier. This notebook converts hardware capacity to decimal GB before comparing.

### 5b. Formulas

KV bytes per token at FP16:

$$\text{GQA/MHA: } 2 \times L_{\text{attn}} \times h_{kv} \times d_{\text{head}} \times 2 \qquad \text{MLA: } L_{\text{attn}} \times (r_{kv} + d_{\text{rope}}) \times 2$$

Weight memory at precision $p$:

$$W_p = N_{\text{total}} \times \frac{b_p}{8} \times o_p$$

| Precision | $b_p$ (bits/weight) | $o_p$ (scale overhead) |
|---|---|---|
| FP16 | 16 | 1.00 |
| FP8 | 8 | 1.00 |
| Q5_K_M | 5.5 | 1.05 |
| Q4_K_M | 4.5 | 1.05 |
| Q3 | 3.3 | 1.05 |

**Note:** `data_dictionary.md` applies the 1.05 overhead to every precision, including FP16. FP16 has no quantization scales, so this notebook uses 1.00 for FP16 and FP8.

In [ ]:
GIB_TO_GB = 2**30 / 1e9
PRECISIONS = {"fp16": (16.0, 1.00), "fp8": (8.0, 1.00), "q5": (5.5, 1.05), "q4": (4.5, 1.05), "q3": (3.3, 1.05)}


def rel_err(stated, expected):
    return abs(stated - expected) / expected if expected else np.nan


# ---- 5c. KV bytes per token from architecture ------------------------------
owm = DFS["open_weight_models"].copy()


def expected_kv_bytes(r):
    attn_layers = r.num_layers - r.linear_attention_layers
    if r.attention_type == "MLA":
        return attn_layers * (r.kv_lora_rank + r.qk_rope_head_dim) * 2
    if r.attention_type in ("GQA", "MHA"):
        return 2 * attn_layers * r.num_kv_heads * r.head_dim * 2
    return np.nan


owm["kv_expected"] = owm.apply(expected_kv_bytes, axis=1)
kv_bad = owm[(owm.kv_expected - owm.kv_bytes_per_token_fp16).abs() > 0.01 * owm.kv_expected]
for _, r in kv_bad.iterrows():
    record("physics", "open_weight_models", "ERROR",
           f"{r.model_key}: kv_bytes_per_token_fp16={r.kv_bytes_per_token_fp16:,.0f}, "
           f"architecture gives {r.kv_expected:,.0f}", [r.model_key])

# ---- 5d. Parameter sanity --------------------------------------------------
for name, df in [("open_weight_models", owm), ("open_weights_ram_residency", DFS["open_weights_ram_residency"])]:
    bad = df[df.active_params_b > df.total_params_b]
    if len(bad):
        record("physics", name, "ERROR", "active_params_b > total_params_b", bad.model_key.tolist())
dense_col = "architecture" if "architecture" in owm else None
dense = owm[(owm.architecture == "dense") & (owm.active_params_b != owm.total_params_b)]
if len(dense):
    record("physics", "open_weight_models", "ERROR", "Dense model with active != total", dense.model_key.tolist())

# Checkpoint size vs params x native dtype bytes.
DTYPE_BYTES = {"FP8": 1.0, "BF16": 2.0, "FP16": 2.0, "FP4": 0.5}
owm["ckpt_expected_gb"] = owm.total_params_b * owm.native_dtype.map(DTYPE_BYTES)
ck_bad = owm[(owm.official_weight_size_gb - owm.ckpt_expected_gb).abs() > REL_TOL * owm.ckpt_expected_gb]
for _, r in ck_bad.iterrows():
    record("physics", "open_weight_models", "WARN",
           f"{r.model_key}: checkpoint {r.official_weight_size_gb} GB vs {r.ckpt_expected_gb:.0f} GB "
           f"expected for {r.total_params_b}B params at {r.native_dtype}", [r.model_key])

# ---- 5e. Residency table: weights re-derived from params -------------------
rr = DFS["open_weights_ram_residency"].copy()
weight_rows = []
for _, r in rr.iterrows():
    for p, (bits, ovh) in PRECISIONS.items():
        stated = r[f"weights_{p}_gb"]
        expected = r.total_params_b * bits / 8 * ovh
        weight_rows.append({"model_key": r.model_key, "precision": p, "stated_gb": stated,
                            "expected_gb": round(expected, 1), "rel_err_pct": round(100 * rel_err(stated, expected), 1),
                            "overhead_gb": r[f"total_{p}_gb"] - stated})
weights_check = pd.DataFrame(weight_rows)
for _, r in weights_check[weights_check.rel_err_pct > 100 * REL_TOL].iterrows():
    record("physics", "open_weights_ram_residency", "WARN",
           f"{r.model_key} {r.precision}: weights {r.stated_gb} GB vs {r.expected_gb} GB from params "
           f"({r.rel_err_pct}% off)", [r.model_key])
for _, r in weights_check[weights_check.overhead_gb < 0].iterrows():
    record("physics", "open_weights_ram_residency", "ERROR",
           f"{r.model_key} {r.precision}: total < weights", [r.model_key])

# KV + runtime overhead does not depend on weight precision. It must be constant per model.
ovh_spread = weights_check.groupby("model_key").overhead_gb.agg(["min", "max"])
ovh_spread = ovh_spread[(ovh_spread["max"] - ovh_spread["min"]) > 1.0]
for mk, r in ovh_spread.iterrows():
    record("physics", "open_weights_ram_residency", "WARN",
           f"{mk}: (total - weights) varies from {r['min']:.0f} to {r['max']:.0f} GB across precisions; "
           "KV + runtime overhead should not change with weight precision", [mk])

print("Residency weights: stated vs re-derived (largest errors)")
display(weights_check.sort_values("rel_err_pct", ascending=False).head(12))

In [ ]:
# ---- 5f. Fit flags and minimum tier ----------------------------------------
# A tier fits 'yes' if total <= USABLE_FRACTION x capacity, 'tight' if <= capacity, else 'no'.
# USABLE_FRACTION leaves room for the OS and other processes.
USABLE_FRACTION = 0.90
TIERS_GIB = {"32gb": 32, "64gb": 64, "128gb": 128, "256gb": 256, "512gb": 512, "1tb": 1024}
LADDER_GIB = {**{k.upper(): v for k, v in TIERS_GIB.items()}, "2TB": 2048, "4TB": 4096}


def fit_label(total_gb, tier_gib):
    cap = tier_gib * GIB_TO_GB
    if total_gb <= USABLE_FRACTION * cap:
        return "yes"
    if total_gb <= cap:
        return "tight"
    return "no"


def min_tier(total_gb):
    for label, gib in LADDER_GIB.items():
        if total_gb <= gib * GIB_TO_GB:
            return label
    return ">4TB"


def norm_tier(label):
    return str(label).upper().replace("+", "").replace(" ", "")


fit_rows = []
for _, r in rr.iterrows():
    for tier, gib in TIERS_GIB.items():
        stated = str(r[f"fits_{tier}"]).lower()
        exp = fit_label(r.total_q4_gb, gib)
        if stated != exp:
            fit_rows.append({"model_key": r.model_key, "tier": tier, "total_q4_gb": r.total_q4_gb,
                             "capacity_gb": round(gib * GIB_TO_GB, 1), "stated": stated, "expected": exp})
    exp_min = min_tier(r.total_q4_gb)
    if norm_tier(r.min_ram_tier_q4) != norm_tier(exp_min):
        record("physics", "open_weights_ram_residency", "WARN",
               f"{r.model_key}: min_ram_tier_q4='{r.min_ram_tier_q4}' but total_q4={r.total_q4_gb} GB "
               f"gives '{exp_min}'", [r.model_key])
fit_mismatch = pd.DataFrame(fit_rows)
for _, r in fit_mismatch.iterrows():
    sev = "ERROR" if (r.stated in ("yes", "tight") and r.expected == "no") else "WARN"
    record("physics", "open_weights_ram_residency", sev,
           f"{r.model_key} @ {r.tier}: stated '{r.stated}', expected '{r.expected}' "
           f"({r.total_q4_gb} GB vs {r.capacity_gb} GB capacity)", [r.model_key])
print(f"Fit-flag mismatches: {len(fit_mismatch)}")
if len(fit_mismatch):
    display(fit_mismatch)

In [ ]:
# ---- 5g. TCO identities (hardware_tco_modeled) -----------------------------
ASM = DFS["assumptions"].set_index("parameter")["value"].astype(float)
tco = DFS["hardware_tco_modeled"].merge(DFS["hardware"][["hw_key", "price_usd"]], on="hw_key", how="left")

exp_dep = tco.price_usd * (1 - ASM["hardware_salvage_value_pct"]) / ASM["hardware_useful_life_months"]
exp_fin = tco.price_usd * ASM["cost_of_capital_annual"] / 12
exp_total = (tco.monthly_depreciation_usd + tco.monthly_financing_usd
             + tco.monthly_electricity_usd + tco.monthly_admin_infra_usd)
exp_tokens = tco.utilization_pct / 100 * 730 * 3600 * tco.decode_tps / 1e6
exp_cost = tco.total_monthly_tco_usd / tco.monthly_tokens_delivered_m

identities = {
    "purchase_price matches hardware.csv": (tco.purchase_price_usd, tco.price_usd),
    "depreciation = price x (1 - salvage) / life": (tco.monthly_depreciation_usd, exp_dep),
    "financing = price x WACC / 12": (tco.monthly_financing_usd, exp_fin),
    "total = sum of components": (tco.total_monthly_tco_usd, exp_total),
    "tokens = U x 730h x 3600 x tps": (tco.monthly_tokens_delivered_m, exp_tokens),
    "cost/Mtok = total / tokens": (tco.cost_per_mtok_output_usd, exp_cost),
}
for label, (stated, expected) in identities.items():
    bad = tco[(stated - expected).abs() > 0.01 + 0.01 * expected.abs()]
    if len(bad):
        record("physics", "hardware_tco_modeled", "ERROR", f"Identity fails: {label}",
               (bad.hw_key + "@" + bad.utilization_pct.astype(str)).tolist())

record("physics", "hardware_tco_modeled", "WARN",
       "decode_tps is the mean aggregate_tps of ALL models measured on that hardware "
       "(build_model.py). Cost/Mtok is therefore not specific to any model.")
record("physics", "hardware_tco_modeled", "INFO",
       "Network cost ($30 desktop / $150 server per month) is hard-coded in build_model.py, "
       "not in assumptions.csv.")

# ---- 5h. Task-cost identities (task_costs_modeled) -------------------------
tc = DFS["task_costs_modeled"]
human_cost = ASM["human_intervention_minutes_per_failure"] / 60 * ASM["human_debugger_hourly_rate"]
checks = {
    "cost_per_successful = cost_per_attempt / p": (tc.cost_per_successful_task_usd, tc.cost_per_attempt_usd / tc.task_success_rate),
    f"human_overhead = (1 - p) x ${human_cost:.0f}": (tc.human_debugging_overhead_usd, (1 - tc.task_success_rate) * human_cost),
    "fully_loaded = cost_per_successful + human_overhead": (tc.fully_loaded_task_cost_usd, tc.cost_per_successful_task_usd + tc.human_debugging_overhead_usd),
}
for label, (stated, expected) in checks.items():
    bad = tc[(stated - expected).abs() > 0.01 + 0.02 * expected.abs()]
    if len(bad):
        record("physics", "task_costs_modeled", "ERROR", f"Identity fails: {label}",
               (bad.workload_id + "/" + bad.model_key).tolist())

# ---- 5i. Price-card sanity -------------------------------------------------
for name in ("api_prices", "api_prices_hosted"):
    df = DFS[name]
    rules = {
        "cached_input > input": df.cached_input_usd_mtok > df.input_usd_mtok,
        "batch_input > input": df.batch_input_usd_mtok > df.input_usd_mtok,
        "batch_output > output": df.batch_output_usd_mtok > df.output_usd_mtok,
        "output < input": df.output_usd_mtok < df.input_usd_mtok,
    }
    for label, mask in rules.items():
        if mask.any():
            record("physics", name, "ERROR" if "output < input" not in label else "WARN",
                   f"Price rule violated: {label}", df.loc[mask, "model_key"].tolist())

show_findings("physics")

## 6. Cross-dataset consistency

The same fact often appears in several files. Each fact must have one value.

In [ ]:
def parse_params_display(text):
    # '2.8T (104B active)' -> (2800.0, 104.0); '685B (37B active)' -> (685.0, 37.0)
    t = str(text)
    m = re.match(r"\s*([\d.]+)\s*([TB])", t)
    if not m:
        return (np.nan, np.nan)
    total = float(m.group(1)) * (1000 if m.group(2) == "T" else 1)
    a = re.search(r"([\d.]+)\s*B\s*active", t)
    return (total, float(a.group(1)) if a else total)


# ---- 6a. Parameter counts --------------------------------------------------
ap = DFS["api_prices"].copy()
ap[["disp_total_b", "disp_active_b"]] = ap.params_display.apply(lambda s: pd.Series(parse_params_display(s)))
params = (rr[["model_key", "total_params_b", "active_params_b"]].rename(columns=lambda c: c if c == "model_key" else f"residency_{c}")
          .merge(owm[["model_key", "total_params_b", "active_params_b"]].rename(columns=lambda c: c if c == "model_key" else f"arch_{c}"), on="model_key", how="outer")
          .merge(ap[["model_key", "disp_total_b", "disp_active_b"]], on="model_key", how="left"))
for _, r in params.iterrows():
    for kind in ("total", "active"):
        vals = {src: r[f"{src}_{kind}_params_b"] if src != "disp" else r[f"disp_{kind}_b"] for src in ("residency", "arch", "disp")}
        vals = {k: v for k, v in vals.items() if pd.notna(v)}
        if len(vals) > 1 and (max(vals.values()) - min(vals.values())) > 0.02 * min(vals.values()):
            record("consistency", "params", "ERROR", f"{r.model_key} {kind} params disagree: {vals}", [r.model_key])

# ---- 6b. Q4 memory footprint ----------------------------------------------
qa = DFS["quant_artifacts"]
q4_qa = qa[qa.quant_name.str.upper().str.startswith("Q4")].groupby("model_key").measured_resident_gb.max()
cm = DFS["chinese_models_hardware_performance_matrix"].copy()
cm["model_key"] = cm.model.map(lambda m: NAME_TO_KEY.get(norm_key(m)))
cm_q4 = cm[cm.quantization.str.contains("Q4|4-bit", case=False, regex=True)]
fp_cols = {
    "residency_total_q4": rr.set_index("model_key").total_q4_gb,
    "quant_artifacts_q4_resident": q4_qa,
    "cn_matrix_q4_min": cm_q4.groupby("model_key").total_memory_req_gb.min(),
    "cn_matrix_q4_max": cm_q4.groupby("model_key").total_memory_req_gb.max(),
    "build_model_q4_weights": owm.set_index("model_key").total_params_b * 4.5 / 8 * 1.05,
}
q4_compare = pd.DataFrame(fp_cols)
q4_compare = q4_compare[q4_compare.notna().sum(axis=1) >= 2].round(1)
q4_compare["spread_pct"] = (100 * (q4_compare.max(axis=1) - q4_compare.min(axis=1)) / q4_compare.min(axis=1)).round(1)
for mk, r in q4_compare[q4_compare.spread_pct > 10].iterrows():
    vals = {k: v for k, v in r.drop("spread_pct").items() if pd.notna(v)}
    record("consistency", "q4_footprint", "WARN", f"{mk}: Q4 footprint differs by {r.spread_pct}% across files: {vals}", [mk])
print("Q4 footprint across datasets (GB):")
display(q4_compare.sort_values("spread_pct", ascending=False))

# ---- 6c. SWE-bench Pro score ----------------------------------------------
bm = DFS["benchmarks"]
swe = bm[bm.benchmark == "SWE-bench Pro"].groupby("model_key").score.agg(lambda s: sorted(set(s)))
swe_cmp = pd.DataFrame({
    "benchmarks_csv": swe,
    "residency": rr.set_index("model_key").swe_bench_pro,
    "cn_matrix": cm.groupby("model_key").swe_bench_pro_pct.agg(lambda s: sorted(set(s))),
}).dropna(how="all")
for mk, r in swe_cmp.iterrows():
    vals = []
    for v in r.values:
        if isinstance(v, list):
            vals.extend(v)
        elif pd.notna(v):
            vals.append(v)
    if len(vals) > 1 and max(vals) - min(vals) > 0.5:
        record("consistency", "swe_bench_pro", "WARN", f"{mk}: SWE-bench Pro values differ: {r.to_dict()}", [mk])
    if isinstance(r.benchmarks_csv, list) and len(r.benchmarks_csv) > 1:
        record("consistency", "swe_bench_pro", "WARN", f"{mk}: several SWE-bench Pro scores in benchmarks.csv {r.benchmarks_csv}", [mk])
in_res_not_bm = sorted(set(rr.model_key[rr.swe_bench_pro.notna()]) - set(swe.index))
if in_res_not_bm:
    record("consistency", "swe_bench_pro", "WARN",
           "SWE-bench Pro score in residency table has no row in benchmarks.csv (unsourced score)", in_res_not_bm)

# ---- 6d. Release dates -----------------------------------------------------
rd = ap[["model_key", "release_date"]].merge(owm[["model_key", "release_date"]], on="model_key", suffixes=("_api", "_arch"))
rd_bad = rd[rd.release_date_api != rd.release_date_arch]
for _, r in rd_bad.iterrows():
    record("consistency", "release_date", "WARN",
           f"{r.model_key}: api_prices {r.release_date_api} vs open_weight_models {r.release_date_arch}", [r.model_key])

show_findings("consistency")

## 7. Generate L2 datasets

This section rebuilds the residency table from curated inputs with the formulas in Section 5.
It writes to `data/generated/` and does not overwrite `data/open_weights_ram_residency.csv`.

$$\text{Total}_p = W_p + \underbrace{\text{KV}_{\text{FP16}}(C)}_{\text{context } C} + R$$

* $C$ = 32,768 tokens, single stream, FP16 KV cache.
* $R$ = runtime overhead (framework, activations, buffers).
* If a model has no architecture row, KV uses `DEFAULT_KV_GB` and the row is marked `kv_source = assumed`.

**Inputs:** `total_params_b`, `active_params_b` (residency table, curated) and `kv_bytes_per_token_fp16` (`open_weight_models`).
After review, point the analysis notebook at `data/generated/open_weights_ram_residency.csv`.

In [ ]:
CONTEXT_TOKENS = 32_768
RUNTIME_OVERHEAD_GB = 6.0
DEFAULT_KV_GB = 8.0

kv_by_model = owm.set_index("model_key").kv_bytes_per_token_fp16
src_col = ["source_id"] if "source_id" in rr.columns else []
inputs = rr[["model_key", "model_name", "lab", "arch_structure", "total_params_b", "active_params_b",
             "attention_mech", "swe_bench_pro", "optimal_target_hardware"] + src_col + ["notes"]].copy()

gen = inputs.copy()
gen["kv_source"] = np.where(gen.model_key.isin(kv_by_model.index), "architecture", "assumed")
gen["kv_gb"] = np.where(gen.kv_source == "architecture",
                        gen.model_key.map(kv_by_model) * CONTEXT_TOKENS / 1e9, DEFAULT_KV_GB).round(2)
for p, (bits, ovh) in PRECISIONS.items():
    gen[f"weights_{p}_gb"] = (gen.total_params_b * bits / 8 * ovh).round(1)
    gen[f"total_{p}_gb"] = (gen[f"weights_{p}_gb"] + gen.kv_gb + RUNTIME_OVERHEAD_GB).round(1)
gen["min_ram_tier_q4"] = gen.total_q4_gb.map(min_tier)
for tier, gib in TIERS_GIB.items():
    gen[f"fits_{tier}"] = gen.total_q4_gb.map(lambda t, g=gib: fit_label(t, g))
gen["derivation"] = (f"W=N*b/8*o; KV@{CONTEXT_TOKENS} FP16; R={RUNTIME_OVERHEAD_GB}GB; "
                     f"usable={USABLE_FRACTION}; capacity in GiB->GB")

col_order = list(rr.columns)
extra = [c for c in gen.columns if c not in col_order]
gen = gen[[c for c in col_order if c in gen.columns] + extra]

# What moves relative to the hand-typed table?
diff = rr[["model_key", "total_q4_gb", "min_ram_tier_q4"]].merge(
    gen[["model_key", "total_q4_gb", "min_ram_tier_q4", "kv_source"]], on="model_key", suffixes=("_current", "_generated"))
diff["delta_q4_gb"] = (diff.total_q4_gb_generated - diff.total_q4_gb_current).round(1)
diff["tier_changed"] = diff.min_ram_tier_q4_current.map(norm_tier) != diff.min_ram_tier_q4_generated.map(norm_tier)
for _, r in diff[diff.tier_changed].iterrows():
    record("generate", "open_weights_ram_residency", "WARN",
           f"{r.model_key}: min tier changes {r.min_ram_tier_q4_current} -> {r.min_ram_tier_q4_generated} when derived", [r.model_key])
print("Current (hand-typed) vs generated Q4 residency:")
display(diff.sort_values("delta_q4_gb", key=abs, ascending=False))

if WRITE_OUTPUTS:
    GEN_DIR.mkdir(exist_ok=True)
    gen.to_csv(GEN_DIR / "open_weights_ram_residency.csv", index=False)
    gen.to_csv(DATA_DIR / "open_weights_ram_residency.csv", index=False)
    q4_compare.to_csv(GEN_DIR / "q4_footprint_cross_check.csv")
    print("Wrote", GEN_DIR / "open_weights_ram_residency.csv")
    print("Synced derived residency table to", DATA_DIR / "open_weights_ram_residency.csv")

## Limits of this notebook

These checks prove **internal** consistency: keys join, formulas reproduce, and files agree with each other.
They do **not** prove that a value matches the outside world. A wrong parameter count that appears in every file passes every check.

To close that gap, each L0 fact needs a verification record: the URL, the exact quoted figure, the date checked, and who checked it.
Until that record exists, treat `confidence = high` as unverified.

## 8. Summary and gate

Set `STRICT = True` in the setup cell to stop the run when an `ERROR` exists.
Use this mode before you publish charts from the analysis notebook.

In [ ]:
findings = pd.DataFrame(FINDINGS)
if findings.empty:
    print("No findings. All checks pass.")
else:
    summary = (findings.pivot_table(index="dataset", columns="severity", values="message",
                                    aggfunc="count", fill_value=0)
               .reindex(columns=["ERROR", "WARN", "INFO"], fill_value=0)
               .sort_values(["ERROR", "WARN"], ascending=False))
    display(summary)
    print("Totals:", findings.severity.value_counts().to_dict())
    if WRITE_OUTPUTS:
        GEN_DIR.mkdir(exist_ok=True)
        findings.to_csv(GEN_DIR / "validation_findings.csv", index=False)
        print("Wrote", GEN_DIR / "validation_findings.csv")
    n_err = int((findings.severity == "ERROR").sum())
    if STRICT and n_err:
        raise RuntimeError(f"{n_err} ERROR findings. Fix the data before running the analysis notebook.")